# 5단계: 밸류에이션 — 지금 코웨이 주식은 싼가 비싼가?

---
### 이 실습에서 배우는 것
- PER, PBR, EV/EBITDA 계산
- 시나리오별 적정주가 계산
- yfinance로 실시간 주가 연동
- 투자 판단 자동화 (조건문 활용)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import platform

if platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
elif platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 120

df = pd.read_csv('data/coway_annual.csv')
for col in ['매출액', '영업이익', '법인세전이익', '당기순이익']:
    df[col] = df[col] / 100  # 억원

# 2025년 기준 데이터
latest = df[df['연도'] == 2025].iloc[0]
주식수 = int(latest['주식수'])

print(f'발행 주식수: {주식수:,}주')
print(f'2025 매출액: {latest["매출액"]:,.0f}억원')
print(f'2025 영업이익: {latest["영업이익"]:,.0f}억원')
print(f'2025 순이익: {latest["당기순이익"]:,.0f}억원')

## 5-1. 주가 데이터 가져오기

In [ ]:
# 방법 1: yfinance로 실시간 주가 (인터넷 필요)
try:
    import yfinance as yf
    ticker = yf.Ticker('021240.KS')  # 코웨이 종목코드
    hist = ticker.history(period='1y')
    현재주가 = hist['Close'].iloc[-1]
    print(f'실시간 주가: {현재주가:,.0f}원 (yfinance)')
except Exception as e:
    print(f'yfinance 오류: {e}')
    # 방법 2: 직접 입력 (인터넷 없을 때)
    현재주가 = 62000  # 분석 시점 주가를 직접 입력
    print(f'수동 입력 주가: {현재주가:,}원')

시가총액_억원 = (현재주가 * 주식수) / 1e8
print(f'시가총액: {시가총액_억원:,.0f}억원 ({시가총액_억원/10000:.1f}조원)')

## 5-2. 핵심 밸류에이션 지표

In [ ]:
# EPS (주당순이익)
순이익_원 = latest['당기순이익'] * 1e8  # 억원 → 원
EPS = 순이익_원 / 주식수

# PER (주가수익비율)
PER = 현재주가 / EPS

# BPS 계산 (자본총계 필요 — 별도 데이터)
# 코웨이 2025 자본총계 약 1.8조원 추정
자본총계_억원 = 18000  # 억원 (근사값)
BPS = (자본총계_억원 * 1e8) / 주식수
PBR = 현재주가 / BPS

# 배당수익률 (2024년 DPS 약 3,700원 기준)
DPS = 3700  # 원
배당수익률 = DPS / 현재주가 * 100

print('=' * 40)
print('     코웨이 밸류에이션 지표')
print('=' * 40)
print(f'현재 주가:     {현재주가:>10,.0f}원')
print(f'EPS:          {EPS:>10,.0f}원')
print(f'BPS:          {BPS:>10,.0f}원')
print(f'DPS:          {DPS:>10,.0f}원')
print('-' * 40)
print(f'PER:          {PER:>10.1f}배')
print(f'PBR:          {PBR:>10.1f}배')
print(f'배당수익률:    {배당수익률:>9.1f}%')
print('=' * 40)
print()
print('[참고 벤치마크]')
print('렌탈/구독업 평균 PER: 12~18배')
print('코스피 평균 PER: 약 10~12배')
print('배당주 기준 수익률: 3% 이상')

## 5-3. 시나리오별 적정주가

In [ ]:
# PER 기반 적정주가 시나리오
시나리오 = {
    '보수적 (PER 10배)': 10,
    '기본 (PER 13배)': 13,
    '낙관적 (PER 16배)': 16,
    '고성장 (PER 20배)': 20,
}

print(f'현재 EPS: {EPS:,.0f}원 기준')
print()
print(f'{"시나리오":<20} {"적정주가":>10} {"현재가 대비":>12} {"판단":>6}')
print('-' * 55)

for name, per_multiple in 시나리오.items():
    적정주가 = EPS * per_multiple
    updown = (적정주가 / 현재주가 - 1) * 100
    판단 = '🔴 고평가' if updown < -10 else '🟢 저평가' if updown > 10 else '🟡 적정'
    print(f'{name:<20} {적정주가:>10,.0f}원 {updown:>+11.1f}% {판단:>6}')

print()
print('* EPS 성장을 반영한 Forward PER도 함께 봐야 합니다')

In [ ]:
# Forward EPS 예측 (EPS 성장률 가정)
현재_EPS = EPS
성장률_시나리오 = [0.05, 0.10, 0.15]  # 5%, 10%, 15% EPS 성장 가정
기준_PER = 13

print('EPS 성장률 가정별 1년 후 적정주가 (PER 13배 기준):')
print()
print(f'{"EPS 성장률":<15} {"Forward EPS":>12} {"1년 후 적정주가":>15} {"현재가 대비":>12}')
print('-' * 58)

for 성장률 in 성장률_시나리오:
    forward_EPS = 현재_EPS * (1 + 성장률)
    forward_주가 = forward_EPS * 기준_PER
    updown = (forward_주가 / 현재주가 - 1) * 100
    print(f'{성장률*100:>6.0f}%          {forward_EPS:>12,.0f}원 {forward_주가:>15,.0f}원 {updown:>+11.1f}%')

## 5-4. 역사적 PER 밴드 차트

In [ ]:
# 연도별 EPS 계산
df['EPS'] = (df['당기순이익'] * 1e8) / df['주식수']

# 연도별 PER 밴드 (PER 8배 ~ 20배)
per_밴드 = [8, 12, 15, 18, 20]
colors_band = ['#E3F2FD', '#BBDEFB', '#90CAF9', '#42A5F5', '#1565C0']

fig, ax = plt.subplots(figsize=(11, 6))

for per, color in zip(per_밴드, colors_band):
    ax.plot(df['연도'], df['EPS'] * per, '--', color=color,
           alpha=0.8, linewidth=1.5, label=f'PER {per}배')
    ax.fill_between(df['연도'], df['EPS'] * per, alpha=0.05, color=color)

# 현재 주가 표시 (가로선)
ax.axhline(y=현재주가, color='red', linewidth=2, linestyle='-',
          label=f'현재주가 {현재주가:,}원')
ax.annotate(f'현재주가\n{현재주가:,}원',
           xy=(df['연도'].max(), 현재주가),
           xytext=(10, 0), textcoords='offset points',
           color='red', fontweight='bold', fontsize=9)

ax.set_title('코웨이 EPS 기반 PER 밴드 차트', fontsize=13, fontweight='bold')
ax.set_ylabel('주가 (원)')
ax.set_xlabel('연도')
ax.legend(loc='upper left')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('data/chart_per_band.png', dpi=150, bbox_inches='tight')
plt.show()
print('→ 현재 주가가 어느 PER 밴드 사이에 있는지 확인하세요')

---
## ✏️ 과제

1. 배당할인모형(DDM)으로 적정주가를 구해보세요.
   ```
   공식: 적정주가 = DPS / (요구수익률 - 배당성장률)
   가정: DPS=3,700원, 요구수익률=8%, 배당성장률=5%
   ```
2. 코스피 평균 PER(약 11배)과 비교했을 때 코웨이는 프리미엄이 붙어야 할까요, 디스카운트여야 할까요? 이유를 설명해보세요.
3. 미국 법인이 흑자전환(2025년)했을 때 EPS 상승 효과를 계산해보세요.